## DATA FOR LSTM

In [21]:
import pandas as pd, numpy as np
df = pd.read_csv("/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/data/card_transaction_v1.csv")

In [22]:
df

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24386895,1999,1,2020,2,27,22:23,$-54.00,Chip Transaction,-5162038175624867091,Merrimack,NH,3054.0,5541,NaN,No
24386896,1999,1,2020,2,27,22:24,$54.00,Chip Transaction,-5162038175624867091,Merrimack,NH,3054.0,5541,NaN,No
24386897,1999,1,2020,2,28,07:43,$59.15,Chip Transaction,2500998799892805156,Merrimack,NH,3054.0,4121,NaN,No
24386898,1999,1,2020,2,28,20:10,$43.12,Chip Transaction,2500998799892805156,Merrimack,NH,3054.0,4121,NaN,No


In [23]:
df.User.value_counts()

User
486     82355
396     80749
332     70010
262     68089
1249    65644
        ...  
457        25
231        21
1367       20
1767       16
1817       15
Name: count, Length: 2000, dtype: int64

In [24]:
user_less_than_14_transactions = df[
    df.groupby("User")["User"].transform("size") < 14
]

In [25]:
user_less_than_14_transactions.shape

(0, 15)

In [26]:
USER_COL = "User"

tmp = df.reset_index(names="original_index").copy()

tmp["run_id"] = tmp[USER_COL].ne(tmp[USER_COL].shift()).cumsum()

continuous_user_runs = (
    tmp.groupby(["run_id", USER_COL], dropna=False)
    .agg(
        start_index=("original_index", "first"),
        end_index=("original_index", "last"),
        continuous_rows=(USER_COL, "size")
    )
    .reset_index()
)

continuous_user_runs

,run_id,User,start_index,end_index,continuous_rows
0,1,0,0,19962,19963
1,2,1,19963,28881,8919
2,3,2,28882,70859,41978
3,4,3,70860,80976,10117
4,5,4,80977,99518,18542
...,...,...,...,...,...
1995,1996,1995,24322025,24336737,14713
1996,1997,1996,24336738,24354749,18012
1997,1998,1997,24354750,24376356,21607
1998,1999,1998,24376357,24382138,5782


In [27]:
user_continuity_summary = (
    continuous_user_runs.groupby(USER_COL, dropna=False)
    .agg(
        num_blocks=("continuous_rows", "size"),
        max_continuous_rows=("continuous_rows", "max"),
        total_rows=("continuous_rows", "sum"),
        avg_continuous_rows=("continuous_rows", "mean")
    )
    .sort_values("max_continuous_rows", ascending=False)
)

user_continuity_summary

,num_blocks,max_continuous_rows,total_rows,avg_continuous_rows
User,,,,
486,1,82355,82355,82355.0
396,1,80749,80749,80749.0
332,1,70010,70010,70010.0
262,1,68089,68089,68089.0
1249,1,65644,65644,65644.0
...,...,...,...,...
457,1,25,25,25.0
231,1,21,21,21.0
1367,1,20,20,20.0


## SPARKOV DATASET

In [28]:
import pandas as pd, numpy as np
sparkov_df = pd.read_csv("/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/data/Simulated_Credit_Card_Sparkov/fraudTrain.csv")

In [29]:
sparkov_df

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1296670,1296670,2020-06-21 12:12:08,30263540414123,fraud_Reichel Inc,entertainment,15.56,Erik,Patterson,M,162 Jessica Row Apt. 072,...,37.7175,-112.4777,258,Geoscientist,1961-11-24,440b587732da4dc1a6395aba5fb41669,1371816728,36.841266,-111.690765,0
1296671,1296671,2020-06-21 12:12:19,6011149206456997,fraud_Abernathy and Sons,food_dining,51.70,Jeffrey,White,M,8617 Holmes Terrace Suite 651,...,39.2667,-77.5101,100,"Production assistant, television",1979-12-11,278000d2e0d2277d1de2f890067dcc0a,1371816739,38.906881,-78.246528,0
1296672,1296672,2020-06-21 12:12:32,3514865930894695,fraud_Stiedemann Ltd,food_dining,105.93,Christopher,Castaneda,M,1632 Cohen Drive Suite 639,...,32.9396,-105.8189,899,Naval architect,1967-08-30,483f52fe67fabef353d552c1e662974c,1371816752,33.619513,-105.130529,0
1296673,1296673,2020-06-21 12:13:36,2720012583106919,"fraud_Reinger, Weissnat and Strosin",food_dining,74.90,Joseph,Murray,M,42933 Ryan Underpass,...,43.3526,-102.5411,1126,Volunteer coordinator,1980-08-18,d667cdcbadaaed3da3f4020e83591c83,1371816816,42.788940,-103.241160,0


In [30]:
sparkov_df["cc_num"].value_counts()

cc_num
4512828414983801773    3123
571365235126           3123
36722699017270         3119
213112402583773        3117
3545109339866548       3113
                       ... 
501894933032              7
4975457191020             7
6577777028615915          7
4225628813173670          7
180097223252063           7
Name: count, Length: 983, dtype: int64

In [34]:
USER_COL = "cc_num"

tmp = (
    sparkov_df
    .reset_index(names="original_index")
    .sort_values(["unix_time", "original_index"], kind="stable")
    .reset_index(drop=True)
)

tmp["run_id"] = tmp[USER_COL].ne(tmp[USER_COL].shift()).cumsum()

continuous_user_runs = (
    tmp.groupby("run_id", sort=False)
    .agg(
        cc_num=(USER_COL, "first"),
        start_index=("original_index", "first"),
        end_index=("original_index", "last"),
        start_time=("trans_date_trans_time", "first"),
        end_time=("trans_date_trans_time", "last"),
        consecutive_rows=(USER_COL, "size"),
    )
    .reset_index(drop=True)
)


In [36]:
continuous_user_runs

,cc_num,start_index,end_index,start_time,end_time,consecutive_rows
0,2703186189652095,0,0,2019-01-01 00:00:18,2019-01-01 00:00:18,1
1,630423337322,1,1,2019-01-01 00:00:44,2019-01-01 00:00:44,1
2,38859492057661,2,2,2019-01-01 00:00:51,2019-01-01 00:00:51,1
3,3534093764340240,3,3,2019-01-01 00:01:16,2019-01-01 00:01:16,1
4,375534208663984,4,4,2019-01-01 00:03:06,2019-01-01 00:03:06,1
...,...,...,...,...,...,...
1294712,30263540414123,1296670,1296670,2020-06-21 12:12:08,2020-06-21 12:12:08,1
1294713,6011149206456997,1296671,1296671,2020-06-21 12:12:19,2020-06-21 12:12:19,1
1294714,3514865930894695,1296672,1296672,2020-06-21 12:12:32,2020-06-21 12:12:32,1
1294715,2720012583106919,1296673,1296673,2020-06-21 12:13:36,2020-06-21 12:13:36,1


In [35]:
continuous_user_runs["consecutive_rows"].value_counts().sort_index()

consecutive_rows
1    1292763
2       1950
3          4
Name: count, dtype: int64